In [ ]:
!pip install -q roboflow ultralytics pyyaml

import torch

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU found. In Colab, go to Runtime > Change runtime type > T4 GPU > Save.")

    import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))




CUDA available: True
GPU: Tesla T4
CUDA available: True
GPU: Tesla T4


In [ ]:
import os
from getpass import getpass

os.environ["ROBOFLOW_API_KEY"] = getpass("Paste your Roboflow API key: ")


Paste your Roboflow API key: ··········


In [ ]:
# DOWNLOAD DATASETS

import os
from roboflow import Roboflow


rf = Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

datasets = []

# street objects
datasets.append(
    rf.workspace("project-rmhsc")
      .project("street-efsqy")
      .version(1)
      .download("yolov8")
)

# more street objects
datasets.append(
    rf.workspace("v-0dx7u")
      .project("street-ci6oy")
      .version(16)
      .download("yolov8")
)

# street lights
datasets.append(
    rf.workspace("jiawei-zhou-e7d1v")
      .project("street-light-pkozz")
      .version(2)
      .download("yolov8")
)

# chairs
datasets.append(
    rf.workspace("new-workspace-xlnp1")
      .project("chair-ikg5k")
      .version(2)
      .download("yolov8")
)

# bollards
datasets.append(
    rf.workspace("blind-obstacle-detection")
      .project("bollard-i4ydf")
      .version(28)
      .download("yolov8")
)

# benches
datasets.append(
    rf.workspace("training-pictures-lv03o")
      .project("bench-o7ozq")
      .version(2)
      .download("yolov8")
)

# pedal bikes
datasets.append(
    rf.workspace("iwanttosleep")
      .project("bicycle-mlc7a")
      .version(2)
      .download("yolov8")
)

# traffic signs
datasets.append(
    rf.workspace("rana-chevuru")
      .project("street-sign")
      .version(2)
      .download("yolov8")
)

for d in datasets:
    print(d.location)


loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bollard-28 in yolov8:: 100%|██████████| 25704/25704 [00:06<00:00, 4275.27it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Bench-2 in yolov8:: 100%|██████████| 504/504 [00:00<00:00, 12020.73it/s]

loading Roboflow workspace...


...


Extracting Dataset Version Zip to Bicycle-2 in yolov8:: 100%|██████████| 1972/1972 [00:00<00:00, 5473.39it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to Street-sign--2 in yolov8:: 100%|██████████| 3026/3026 [00:00<00:00, 3871.04it/s]

/content/Street-1
/content/street-16
/content/street-light-2
/content/chair-2
/content/bollard-28
/content/Bench-2
/content/Bicycle-2
/content/Street-sign--2


In [ ]:
# MERGE THE DATASETS INTO 8 CATEGORIES THEN MIX THEM UP RANDOMLY

import shutil
import yaml
import random
from pathlib import Path
from uuid import uuid4

random.seed(42)

FINAL_CLASSES = [
    "person",
    "vehicle",
    "bicycle",
    "bench",
    "chair",
    "bollard",
    "street_light",
    "traffic_sign",
]

FINAL_CLASS_TO_ID = {name: i for i, name in enumerate(FINAL_CLASSES)}

CLASS_MAP = {
    # Person
    "person": "person",
    "Person": "person",
    "human": "person",
    "insan": "person",
    "ped": "person",

    # Vehicle
    "car": "vehicle",
    "Car": "vehicle",
    "bus": "vehicle",
    "Bus": "vehicle",
    "truck": "vehicle",
    "Truck": "vehicle",
    "MotorCycle": "vehicle",
    "motorbike": "vehicle",
    "motobike": "vehicle",
    "motorcycle": "vehicle",
    "Two-wheeler": "vehicle",
    "Auto": "vehicle",
    "van": "vehicle",
    "araç": "vehicle",

    # Bicycle
    "bicycle": "bicycle",
    "Bicycle": "bicycle",
    "Bike": "bicycle",

    # Bench
    "bench": "bench",
    "Bench": "bench",
    "bank": "bench",

    # Chair
    "chair": "chair",
    "Chair": "chair",

    # Bollard / sidewalk obstacles
    "bollard": "bollard",
    "Bollard": "bollard",
    "bollard_damage": "bollard",
    "bollard_drop": "bollard",
    "bollard_nomal": "bollard",
    "bollard_nomark": "bollard",
    "Tubular marker": "bollard",
    "Pillar": "bollard",
    "spherical_roadblock": "bollard",

    # Street light, lamp style
    "street light": "street_light",
    "Street light": "street_light",
    "street lamp": "street_light",
    "street_light": "street_light",

    # Traffic signs
    "traffic_sign": "traffic_sign",
    "street_sign": "traffic_sign",
    "Trafik işareti": "traffic_sign",
    "Sign'": "traffic_sign",

    "Barrier Sign": "traffic_sign",
    "Bike Sign": "traffic_sign",
    "Bus Stop Sign": "traffic_sign",
    "Construction Sign": "traffic_sign",
    "Handicap Sign": "traffic_sign",
    "Interstate Sign": "traffic_sign",
    "No Entry Sign": "traffic_sign",
    "No Left Turn Sign": "traffic_sign",
    "No Parking Sign": "traffic_sign",
    "No Turn Right Sign": "traffic_sign",
    "No Turn Sign": "traffic_sign",
    "OneWay Sign": "traffic_sign",
    "Parking Sign": "traffic_sign",
    "Pedestrian Crossing Sign": "traffic_sign",
    "School Sign": "traffic_sign",
    "Speed Limit Sign": "traffic_sign",
    "Stop Sign": "traffic_sign",
    "Street Sign": "traffic_sign",
}

MERGED_DIR = Path("/content/sidewalk_objects_v2_yolov8")

if MERGED_DIR.exists():
    shutil.rmtree(MERGED_DIR)

for split in ["train", "valid", "test"]:
    (MERGED_DIR / split / "images").mkdir(parents=True, exist_ok=True)
    (MERGED_DIR / split / "labels").mkdir(parents=True, exist_ok=True)

def read_yaml(path):
    with open(path, "r") as f:
        return yaml.safe_load(f)

def normalize_names(names):
    if isinstance(names, dict):
        return [names[i] for i in sorted(names.keys())]
    return list(names)

def choose_split():
    r = random.random()
    if r < 0.80:
        return "train"
    elif r < 0.90:
        return "valid"
    else:
        return "test"

def find_image_for_label(image_dir, stem):
    for ext in [".jpg", ".jpeg", ".png", ".bmp", ".webp"]:
        p = image_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

def merge_dataset(dataset_location):
    dataset_path = Path(dataset_location)
    data_yaml_path = dataset_path / "data.yaml"

    if not data_yaml_path.exists():
        print("Skipping, no data.yaml:", dataset_path)
        return

    data_yaml = read_yaml(data_yaml_path)
    source_names = normalize_names(data_yaml["names"])

    print("\nMerging:", dataset_path)
    print("Source classes:", source_names)

    for source_split in ["train", "valid", "test"]:
        image_dir = dataset_path / source_split / "images"
        label_dir = dataset_path / source_split / "labels"

        if not image_dir.exists() or not label_dir.exists():
            continue

        for label_path in label_dir.glob("*.txt"):
            image_path = find_image_for_label(image_dir, label_path.stem)
            if image_path is None:
                continue

            new_lines = []

            with open(label_path, "r") as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue

                    old_class_id = int(float(parts[0]))
                    if old_class_id >= len(source_names):
                        continue

                    old_class_name = source_names[old_class_id]
                    final_class_name = CLASS_MAP.get(old_class_name)

                    # Ignore classes not in our final 8
                    if final_class_name is None:
                        continue

                    final_class_id = FINAL_CLASS_TO_ID[final_class_name]
                    new_lines.append(" ".join([str(final_class_id)] + parts[1:]))

            # Keep only images with at least one useful label
            if not new_lines:
                continue

            out_split = choose_split()
            new_stem = f"{dataset_path.name}_{source_split}_{uuid4().hex}"

            out_image_path = MERGED_DIR / out_split / "images" / f"{new_stem}{image_path.suffix.lower()}"
            out_label_path = MERGED_DIR / out_split / "labels" / f"{new_stem}.txt"

            shutil.copy2(image_path, out_image_path)

            with open(out_label_path, "w") as f:
                f.write("\n".join(new_lines) + "\n")

for d in datasets:
    merge_dataset(d.location)

data_yaml = {
    "path": str(MERGED_DIR),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": FINAL_CLASSES,
}

with open(MERGED_DIR / "data.yaml", "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("\nCreated merged dataset:", MERGED_DIR)
print("data.yaml:", MERGED_DIR / "data.yaml")



Merging: /content/Street-1
Source classes: ['Bike', 'Bollard', 'Bus', 'Car', 'Kickboard', 'MotorCycle', 'Person', 'Pillar', 'Sign-', 'Traffic cone', 'Trash', 'Tree', 'Truck', 'Tubular marker', 'obstacles-yBnK']

Merging: /content/street-16
Source classes: ['Cop', 'Elektrik diregi', 'Trafik isareti', 'agac', 'arac', 'bank', 'insan']

Merging: /content/street-light-2
Source classes: ['street light']

Merging: /content/chair-2
Source classes: ['chair']

Merging: /content/bollard-28
Source classes: ['bollard', 'crosswalk', 'greenlight', 'redlight']

Merging: /content/Bench-2
Source classes: ['Bench']

Merging: /content/Bicycle-2
Source classes: ['Bicycle']

Merging: /content/Street-sign--2
Source classes: ['Barrier Sign', 'Bike Sign', 'Bus Stop Sign', 'Construction Sign', 'Handicap Sign', 'Interstate Sign', 'No Entry Sign', 'No Left Turn Sign', 'No Parking Sign', 'No Turn Right Sign', 'No Turn Sign', 'OneWay Sign', 'Parking Sign', 'Pedestrian Crossing Sign', 'School Sign', 'Speed Limit Si

In [ ]:
# CHECK HOW MANY LABELED EXAMPLES IN EACH CATEGORY
from collections import Counter

for split in ["train", "valid", "test"]:
    counts = Counter()
    label_dir = MERGED_DIR / split / "labels"

    for label_path in label_dir.glob("*.txt"):
        with open(label_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if len(parts) >= 5:
                    class_id = int(float(parts[0]))
                    counts[FINAL_CLASSES[class_id]] += 1

    print("\n", split)
    for cls in FINAL_CLASSES:
        print(f"{cls}: {counts[cls]}")



 train
person: 1312
vehicle: 160
bicycle: 2012
bench: 420
chair: 524
bollard: 6864
street_light: 4075
traffic_sign: 1225

 valid
person: 146
vehicle: 14
bicycle: 292
bench: 71
chair: 87
bollard: 806
street_light: 471
traffic_sign: 177

 test
person: 151
vehicle: 17
bicycle: 314
bench: 44
chair: 76
bollard: 790
street_light: 585
traffic_sign: 190


In [ ]:
# TRAIN THE MODEL

!yolo task=detect mode=train \
  model=yolov8n.pt \
  data=/content/sidewalk_objects_v2_yolov8/data.yaml \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  project=/content/runs \
  name=sidewalk_objects_v2_yolov8


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/sidewalk_objects_v2_yolov8/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=sidewalk_objects

In [ ]:
# CHECK HOW GOOD THE MODEL IS AFTER TRAINING
# It runs the model and prints the result

!yolo task=detect mode=val \
  model=/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt \
  data=/content/sidewalk_objects_v2_yolov8/data.yaml \
  imgsz=640


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1307.5±590.4 MB/s, size: 414.8 KB)
val: Scanning /content/sidewalk_objects_v2_yolov8/valid/labels.cache... 934 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 934/934 150.7Mit/s 0.0s
val: /content/sidewalk_objects_v2_yolov8/valid/images/street-16_train_02c38f95c68c4801b29aefb7687b8eae.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 59/59 4.9it/s 12.0s
                   all        934       2063      0.734      0.689        0.7      0.498
                person         56        145      0.433      0.138      0.168     0.0812
               vehicle         11         14      0.543       0.68      0.539      0.303
               bicycle        106        292      0.716      0.568   

In [ ]:
from pathlib import Path
import shutil
import yaml
from uuid import uuid4

SRC_DIR = Path("/content/sidewalk_objects_v2_yolov8")
BOOST_DIR = Path("/content/sidewalk_objects_v2_person_boost_yolov8")

PERSON_CLASS_ID = 0
DUPLICATES_PER_PERSON_IMAGE = 2  # try 2 first, 3 if person is still weak

if BOOST_DIR.exists():
    shutil.rmtree(BOOST_DIR)

shutil.copytree(SRC_DIR, BOOST_DIR)

train_img_dir = BOOST_DIR / "train" / "images"
train_lbl_dir = BOOST_DIR / "train" / "labels"

image_exts = [".jpg", ".jpeg", ".png", ".bmp", ".webp"]

def find_image(stem):
    for ext in image_exts:
        p = train_img_dir / f"{stem}{ext}"
        if p.exists():
            return p
    return None

person_image_count = 0
new_image_count = 0

for label_path in list(train_lbl_dir.glob("*.txt")):
    with open(label_path, "r") as f:
        lines = f.readlines()

    has_person = any(
        len(line.strip().split()) >= 5 and int(float(line.strip().split()[0])) == PERSON_CLASS_ID
        for line in lines
    )

    if not has_person:
        continue

    image_path = find_image(label_path.stem)
    if image_path is None:
        continue

    person_image_count += 1

    for _ in range(DUPLICATES_PER_PERSON_IMAGE):
        new_stem = f"{label_path.stem}_personboost_{uuid4().hex}"
        shutil.copy2(image_path, train_img_dir / f"{new_stem}{image_path.suffix.lower()}")
        shutil.copy2(label_path, train_lbl_dir / f"{new_stem}.txt")
        new_image_count += 1

# Make sure data.yaml points to boosted folder
data_yaml_path = BOOST_DIR / "data.yaml"
with open(data_yaml_path, "r") as f:
    data = yaml.safe_load(f)

data["path"] = str(BOOST_DIR)

with open(data_yaml_path, "w") as f:
    yaml.safe_dump(data, f, sort_keys=False)

print("Person training images found:", person_image_count)
print("Extra duplicated person images added:", new_image_count)
print("Boosted dataset:", BOOST_DIR)
print("Boosted data.yaml:", data_yaml_path)


Person training images found: 425
Extra duplicated person images added: 850
Boosted dataset: /content/sidewalk_objects_v2_person_boost_yolov8
Boosted data.yaml: /content/sidewalk_objects_v2_person_boost_yolov8/data.yaml


In [ ]:
!yolo task=detect mode=train \
  model=/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt \
  data=/content/sidewalk_objects_v2_person_boost_yolov8/data.yaml \
  epochs=12 \
  imgsz=640 \
  batch=16 \
  project=/content/runs \
  name=sidewalk_objects_v2_person_boost


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/sidewalk_objects_v2_person_boost_yolov8/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=12, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt, momentu

In [ ]:
!yolo task=detect mode=val \
  model=/content/runs/sidewalk_objects_v2_person_boost/weights/best.pt \
  data=/content/sidewalk_objects_v2_person_boost_yolov8/data.yaml \
  imgsz=640


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 1284.5±279.7 MB/s, size: 486.3 KB)
val: Scanning /content/sidewalk_objects_v2_person_boost_yolov8/valid/labels.cache... 934 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 934/934 126.4Mit/s 0.0s
val: /content/sidewalk_objects_v2_person_boost_yolov8/valid/images/street-16_train_02c38f95c68c4801b29aefb7687b8eae.jpg: 1 duplicate labels removed
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 59/59 5.7it/s 10.4s
                   all        934       2063      0.822      0.672      0.708      0.495
                person         56        145      0.761     0.0878      0.169     0.0881
               vehicle         11         14      0.741      0.714       0.61       0.31
               bicycle        106        29

In [ ]:
!yolo mode=export \
  model=/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt \
  format=onnx \
  imgsz=640 \
  dynamic=True \
  simplify=True


Ultralytics 8.4.115 🚀 Python-3.12.13 torch-2.11.0+cu128 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino/
Model summary (fused): 73 layers, 3,007,208 parameters, 0 gradients, 8.1 GFLOPs

PyTorch: starting from '/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt' with input shape (1, 3, 640, 640) BCHW and output shape(s) (1, 12, 8400) (6.0 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 12 packages in 305ms
Prepared 4 packages in 1.77s
Installed 4 packages in 292ms
 + colorama==0.4.6
 + onnx==1.22.0
 + onnxruntime==1.28.0
 + onnxslim==0.1.95

requirements: AutoUpdate success ✅ 3.0s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.22.0 opset 20...
ONNX

In [ ]:
from google.colab import files

files.download("/content/runs/sidewalk_objects_v2_yolov8/weights/best.onnx")
files.download("/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import drive
import shutil
from pathlib import Path

drive.mount("/content/drive")

DEST = Path("/content/drive/MyDrive/sidewalk_yolo_model")
DEST.mkdir(parents=True, exist_ok=True)

shutil.copy2("/content/runs/sidewalk_objects_v2_yolov8/weights/best.onnx", DEST / "best.onnx")
shutil.copy2("/content/runs/sidewalk_objects_v2_yolov8/weights/best.pt", DEST / "best.pt")
shutil.copy2("/content/sidewalk_objects_v2_yolov8/data.yaml", DEST / "data.yaml")

print("Saved to:", DEST)


Mounted at /content/drive
Saved to: /content/drive/MyDrive/sidewalk_yolo_model


In [ ]:
import yaml
from pathlib import Path
from google.colab import files

data_yaml = {
    "path": ".",
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": [
        "person",
        "vehicle",
        "bicycle",
        "bench",
        "chair",
        "bollard",
        "street_light",
        "traffic_sign",
    ],
}

yaml_path = Path("/content/data.yaml")

with open(yaml_path, "w") as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

files.download(str(yaml_path))


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>